# From Notebooks to Apps — Building GenAI Applications

### The bridge between "I can call an API" and "I have a working application"

---

This notebook is the **hands-on companion** to **Section 3** on the course page:

- **Chapter 1:** Error Handling & Retries
- **Chapter 2:** Streaming Responses
- **Chapter 3:** Multi-Provider APIs (OpenAI, Claude, Groq)
- **Chapter 4:** App Architecture
- **Chapter 5:** Build: CLI Tool
- **Chapter 6:** Build: Streamlit App

**What makes this notebook different:** We're not just experimenting — we're building real applications. The notebook teaches the concepts; the `.py` files in `cli_tool/` and `streamlit_app/` are the finished products.

---

## Setup

In [ ]:
!pip install openai python-dotenv -q

In [1]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)

True

In [1]:
import os
import sys
import json
import time
from openai import OpenAI, RateLimitError, APITimeoutError, BadRequestError

if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("Set OPENAI_API_KEY first.")

client = OpenAI()

# Groq client for small model demos
GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
groq_client = None
if GROQ_API_KEY:
    groq_client = OpenAI(api_key=GROQ_API_KEY, base_url="https://api.groq.com/openai/v1")
    print("✅ Both OpenAI and Groq clients ready.")
else:
    print("✅ OpenAI client ready. (Groq not configured — set GROQ_API_KEY for multi-provider demos)")

✅ Both OpenAI and Groq clients ready.


---

# Chapter 1: Error Handling & Retries

---

*Course page: Chapter 1 — Error Handling & Retries*

In a notebook, if an API call fails you just re-run the cell. In production, your app handles it automatically.

### Demo 1: What errors look like

In [2]:
# Let's trigger a real error — wrong model name
try:
    response = client.chat.completions.create(
        model="gpt-5-turbo-does-not-exist",
        messages=[{"role": "user", "content": "Hello"}]
    )
except Exception as e:
    print(f"Error type: {type(e).__name__}")
    print(f"Message: {e}")
    print(f"\n💡 This is a BadRequestError — the model name is wrong.")
    print(f"   This is a BUG, not a transient error. Fix the code, don't retry.")

Error type: NotFoundError
Message: Error code: 404 - {'error': {'message': 'The model `gpt-5-turbo-does-not-exist` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}

💡 This is a BadRequestError — the model name is wrong.
   This is a BUG, not a transient error. Fix the code, don't retry.


In [4]:
# What a context overflow looks like
try:
    # Create a message that's way too long (won't actually fail on mini with 128K context,
    # but demonstrates the pattern)
    long_text = "Hello " * 5000000  # ~50K tokens
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": long_text}],
        max_tokens=500
    )
    print(f"Worked! Input tokens: {response.usage.prompt_tokens}")
    print(f"(gpt-4o-mini has 128K context — hard to overflow)")
except BadRequestError as e:
    print(f"Context overflow: {e}")

RateLimitError: Error code: 429 - {'error': {'message': 'Request too large for gpt-4o-mini in organization org-SBQDxSttDwpLundAxILzN7P8 on tokens per min (TPM): Limit 4000000, Requested 7500002. The input or output tokens must be reduced in order to run successfully. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}

### Demo 2: The retry pattern — exponential backoff

In [5]:
def call_with_retry(prompt, system_prompt=None, model="gpt-4o-mini",
                    temperature=0.3, max_tokens=1000, max_retries=3):
    """Call the API with automatic retry on transient errors.
    
    Retries on: RateLimitError, APITimeoutError
    Does NOT retry on: BadRequestError (that's a code bug)
    """
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})
    
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=messages,
                temperature=temperature,
                max_tokens=max_tokens,
                timeout=30  # 30 second timeout
            )
            return response
        
        except RateLimitError:
            wait = 2 ** attempt  # 1s → 2s → 4s
            print(f"⚠️  Rate limited. Waiting {wait}s... (attempt {attempt + 1}/{max_retries})")
            time.sleep(wait)
        
        except APITimeoutError:
            wait = 2 ** attempt
            print(f"⚠️  Timeout. Waiting {wait}s... (attempt {attempt + 1}/{max_retries})")
            time.sleep(wait)
        
        except BadRequestError as e:
            print(f"❌ Bad request (don't retry): {e}")
            raise
    
    raise Exception(f"Failed after {max_retries} retries.")


# Test it — should work first try
response = call_with_retry("What is Hadoop? One sentence.")
print(response.choices[0].message.content)
print(f"✅ Success on first attempt (no retries needed)")

Hadoop is an open-source framework that enables the distributed processing and storage of large datasets across clusters of computers using simple programming models.
✅ Success on first attempt (no retries needed)


### Demo 3: Checking finish_reason — was the response cut off?

In [6]:
# Force a truncation by setting max_tokens very low
response = call_with_retry(
    "Explain the complete history of distributed computing.",
    max_tokens=20  # Way too short for this question
)

print("Response:", response.choices[0].message.content)
print(f"\nFinish reason: {response.choices[0].finish_reason}")

if response.choices[0].finish_reason == "length":
    print("⚠️  Response was TRUNCATED — hit max_tokens limit!")
    print("   In production: increase max_tokens or tell the user the response was cut off.")
elif response.choices[0].finish_reason == "stop":
    print("✅ Response completed normally.")

Response: Distributed computing is a field of computer science that involves a model in which components located on networked computers

Finish reason: length
⚠️  Response was TRUNCATED — hit max_tokens limit!
   In production: increase max_tokens or tell the user the response was cut off.


### Key takeaway

| Error | Status | Retry? | Fix |
|-------|--------|--------|-----|
| Rate limit | 429 | ✅ Yes (exponential backoff) | Wait and retry |
| Timeout | - | ✅ Yes | Wait and retry, or reduce prompt size |
| Bad request | 400 | ❌ No | Fix your code (wrong model, bad params) |
| Auth error | 401 | ❌ No | Fix your API key |
| Context overflow | 400 | ❌ No | Shorten your prompt |

---

---

# Chapter 2: Streaming Responses

---

*Course page: Chapter 2 — Streaming Responses*

Without streaming: user stares at a blank screen for 3-10 seconds.  
With streaming: tokens appear immediately as they're generated.

### Demo 1: Standard vs streaming — feel the difference

In [7]:
prompt = "Explain MapReduce in 3 sentences."

# Standard — wait for everything
print("STANDARD (wait for full response):")
print("─" * 40)
start = time.time()
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}]
)
elapsed = time.time() - start
print(response.choices[0].message.content)
print(f"\n⏱️  Total wait: {elapsed:.1f}s (user saw NOTHING until this moment)")

STANDARD (wait for full response):
────────────────────────────────────────
MapReduce is a programming model designed for processing and generating large datasets with a distributed algorithm on a cluster. It consists of two main functions: the "Map" function that transforms input data into key-value pairs, and the "Reduce" function that aggregates these pairs to produce the final output. This framework enables efficient data processing by allowing parallel execution and fault tolerance across multiple machines.

⏱️  Total wait: 2.7s (user saw NOTHING until this moment)


In [10]:
# Streaming — tokens appear immediately
print("STREAMING (token by token):")
print("─" * 40)

start = time.time()
first_token_time = None

stream = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    stream=True  # ← only change
)

for chunk in stream:
    # print(chunk)
    content = chunk.choices[0].delta.content
    if content:
        if first_token_time is None:
            first_token_time = time.time() - start
        print(content, end="", flush=True)

elapsed = time.time() - start
print(f"\n\n⏱️  First token appeared in: {first_token_time:.2f}s")
print(f"⏱️  Full response completed in: {elapsed:.1f}s")
print(f"💡 User started reading after {first_token_time:.2f}s instead of waiting {elapsed:.1f}s")

STREAMING (token by token):
────────────────────────────────────────
MapReduce is a programming model used for processing and generating large data sets in a distributed computing environment. It consists of two key functions: the "Map" function, which processes input data and produces key-value pairs, and the "Reduce" function, which takes these key-value pairs and aggregates them to produce the final output. This paradigm allows for efficient parallel processing across multiple machines, making it suitable for handling vast amounts of data in a scalable manner.

⏱️  First token appeared in: 0.98s
⏱️  Full response completed in: 2.4s
💡 User started reading after 0.98s instead of waiting 2.4s


### Demo 2: Collecting streamed content for processing

In [11]:
# In real apps, you need to both DISPLAY and SAVE the streamed response

def stream_and_collect(prompt, system_prompt=None, model="gpt-4o-mini"):
    """Stream response to screen AND collect the full text."""
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})
    
    stream = client.chat.completions.create(
        model=model, messages=messages, stream=True
    )
    
    collected = []
    for chunk in stream:
        content = chunk.choices[0].delta.content
        if content:
            print(content, end="", flush=True)
            collected.append(content)
    
    full_response = "".join(collected)
    print()  # newline after streaming
    return full_response


# Use it
result = stream_and_collect("What are 3 use cases for RAG?")

print(f"\n{'─' * 50}")
print(f"Collected {len(result)} characters for further processing.")
print(f"First 80 chars: {result[:80]}...")

Retrieval-Augmented Generation (RAG) is a method that combines generative models with retrieval mechanisms to enhance the quality and accuracy of the generated content. Here are three popular use cases for RAG:

1. **Customer Support and Chatbots**:
   RAG can be used to power intelligent customer support systems. By retrieving relevant information from a database or knowledge base (such as FAQs, manuals, or previous support tickets) and combining it with generative responses, the system can provide precise and contextually relevant answers to customer inquiries. This improves customer satisfaction by delivering accurate information quickly while minimizing the burden on support staff.

2. **Content Creation and Personalized Recommendations**:
   In content creation, RAG can be used to assist writers, marketers, or educators by retrieving relevant articles, studies, or information from a vast corpus and generating coherent and contextually appropriate content. This can aid in drafting 

### Key takeaway

- `stream=True` is the **only change** in the API call
- Response comes as **chunks** instead of one object — iterate with a for loop
- Each chunk has `chunk.choices[0].delta.content` (note: `delta`, not `message`)
- **No usage object** in streaming — you'll need to estimate tokens or make a separate count
- Use streaming for any response the user will read (conversations, summaries, explanations)
- Skip streaming for machine-processed output (JSON classification, short answers)

---

---

# Chapter 3: Multi-Provider APIs

---

*Course page: Chapter 3 — Multi-Provider APIs*

Same concept (send messages → get response), but each provider has different quirks.

### Demo 1: OpenAI vs Groq — same library, different endpoint

In [12]:
prompt = "What is HDFS? One sentence only."

# OpenAI
print("OpenAI (gpt-4o-mini):")
print("─" * 40)
r1 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}]
)
print(r1.choices[0].message.content)
print(f"Tokens: {r1.usage.total_tokens}")

# Groq (same library!)
if groq_client:
    print(f"\nGroq (llama-3.1-8b):")
    print("─" * 40)
    r2 = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )
    print(r2.choices[0].message.content)
    print(f"Tokens: {r2.usage.total_tokens}")
    
    print(f"\n💡 Same OpenAI library, same code pattern.")
    print(f"   Only difference: client = OpenAI(base_url='https://api.groq.com/openai/v1')")
else:
    print("\n(Groq not configured — set GROQ_API_KEY to try this)")

OpenAI (gpt-4o-mini):
────────────────────────────────────────
HDFS, or Hadoop Distributed File System, is a distributed file storage system designed to handle large volumes of data across clusters of computers while providing high availability and fault tolerance.
Tokens: 50

Groq (llama-3.1-8b):
────────────────────────────────────────
HDFS (Hadoop Distributed File System) is an open-source, distributed, parallel, and scalable file system designed for storing and processing large data sets across a cluster of commodity servers.
Tokens: 82

💡 Same OpenAI library, same code pattern.
   Only difference: client = OpenAI(base_url='https://api.groq.com/openai/v1')


### Demo 2: Building a provider-agnostic wrapper

In [10]:
# A simple wrapper that works with any OpenAI-compatible provider

class LLMClient:
    """Provider-agnostic LLM client.
    
    Works with OpenAI, Groq, and any OpenAI-compatible API.
    In production, you'd add Anthropic support with an adapter.
    """
    
    def __init__(self, provider="openai"):
        if provider == "openai":
            self.client = OpenAI()
            self.default_model = "gpt-4o-mini"
        elif provider == "groq":
            self.client = OpenAI(
                api_key=os.environ.get("GROQ_API_KEY"),
                base_url="https://api.groq.com/openai/v1"
            )
            self.default_model = "llama-3.1-8b-instant"
        else:
            raise ValueError(f"Unknown provider: {provider}")
        
        self.provider = provider
    
    def ask(self, prompt, system_prompt=None, temperature=0.3, model=None):
        """Send a prompt, return the response text."""
        messages = []
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        messages.append({"role": "user", "content": prompt})
        
        response = self.client.chat.completions.create(
            model=model or self.default_model,
            messages=messages,
            temperature=temperature,
            max_tokens=1000
        )
        return response.choices[0].message.content


# Same code, different providers
openai_llm = LLMClient("openai")
print("OpenAI:", openai_llm.ask("What is YARN? One sentence."))

if groq_client:
    groq_llm = LLMClient("groq")
    print("\nGroq:", groq_llm.ask("What is YARN? One sentence."))

print("\n💡 Your application code calls llm.ask() — it doesn't care which provider is behind it.")
print("   Swap providers by changing one line, not rewriting your entire app.")

OpenAI: YARN (Yet Another Resource Negotiator) is a resource management and job scheduling technology used in Hadoop to efficiently manage and allocate system resources for distributed applications.

Groq: YARN (Yet Another Resource Negotiator) is an Apache Hadoop technology that provides a resource management layer for distributed applications, allowing them to run on top of Hadoop clusters.

💡 Your application code calls llm.ask() — it doesn't care which provider is behind it.
   Swap providers by changing one line, not rewriting your entire app.


---

# Chapter 4: App Architecture

---

*Course page: Chapter 4 — App Architecture*

A notebook runs top to bottom. An app has layers — each with a specific job. Let's see the difference.

| | Notebook | Application |
|---|---|---|
| Error handling | Re-run the cell | Automatic retry with backoff |
| User input | Hardcoded in cells | User types in UI |
| State | Lost on kernel restart | Persisted in database/session |
| Secrets | os.environ in notebook | Environment variables / vault |
| Output | print() | Formatted UI components |
| Deployment | Your laptop | Server, cloud, or container |

The next two chapters build real apps that handle all of this.

---

# Chapter 5: Build — CLI Tool

---

*Course page: Chapter 5 — Build: CLI Tool*

We'll build a CLI tool step by step — from the simplest version to the full-featured `ask.py`.

### Version 1: The absolute minimum (10 lines)

In [ ]:
# ============================================================
# V1: Simplest possible CLI — no error handling, no options
# ============================================================
# If this were a .py file, you'd run: python ask_v1.py "What is Hadoop?"

def cli_v1(question):
    """The simplest possible LLM CLI."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": question}]
    )
    print(response.choices[0].message.content)


# Test it
cli_v1("What is Hadoop? One sentence.")

print("\n💡 This works but has zero error handling, no cost tracking, no options.")

### Version 2: Add error handling + cost tracking

In [ ]:
# ============================================================
# V2: Add retry logic + token/cost tracking
# ============================================================

PRICING = {
    "gpt-4o":      {"input": 2.50,  "output": 10.00},
    "gpt-4o-mini": {"input": 0.15,  "output": 0.60},
}

def cli_v2(question, model="gpt-4o-mini"):
    """CLI with error handling and cost tracking."""
    messages = [{"role": "user", "content": question}]
    
    # Retry logic
    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=messages,
                max_tokens=1000,
                timeout=30
            )
            
            # Print response
            print(response.choices[0].message.content)
            
            # Check if truncated
            if response.choices[0].finish_reason == "length":
                print("\n⚠️  Response was truncated.")
            
            # Cost tracking
            usage = response.usage
            pricing = PRICING.get(model, PRICING["gpt-4o-mini"])
            cost = (usage.prompt_tokens * pricing["input"] + 
                    usage.completion_tokens * pricing["output"]) / 1_000_000
            
            print(f"\n{'─' * 50}")
            print(f"Model: {model} | Tokens: {usage.total_tokens} "
                  f"(in {usage.prompt_tokens}, out {usage.completion_tokens}) | "
                  f"Cost: ~${cost:.6f}")
            return
        
        except (RateLimitError, APITimeoutError) as e:
            wait = 2 ** attempt
            print(f"⚠️  {type(e).__name__}. Retrying in {wait}s...")
            time.sleep(wait)
        
        except Exception as e:
            print(f"❌ Error: {e}")
            return
    
    print("❌ Failed after 3 retries.")


# Test it
cli_v2("What is Hadoop? Two sentences max.")

### Version 3: Add streaming

In [ ]:
# ============================================================
# V3: Add streaming support
# ============================================================

def cli_v3(question, model="gpt-4o-mini", stream=True):
    """CLI with streaming + error handling + cost tracking."""
    messages = [{"role": "user", "content": question}]
    
    try:
        if stream:
            # Streaming mode
            response_stream = client.chat.completions.create(
                model=model, messages=messages, max_tokens=1000, stream=True
            )
            collected = []
            for chunk in response_stream:
                content = chunk.choices[0].delta.content
                if content:
                    print(content, end="", flush=True)
                    collected.append(content)
            full_text = "".join(collected)
            est_tokens = len(full_text) // 4 + len(question) // 4
            print(f"\n\n{'─' * 50}")
            print(f"Model: {model} | ~{est_tokens} tokens (estimated)")
        else:
            # Standard mode
            cli_v2(question, model)
    
    except Exception as e:
        print(f"\n❌ Error: {e}")


# Test streaming
print("Streaming:")
cli_v3("Explain YARN in 3 sentences.")

### Version 4: Add system prompt + argparse

This is where we'd normally switch to a `.py` file — argparse works in scripts, not notebooks. But let's simulate the argument parsing:

In [ ]:
# ============================================================
# V4: System prompt support
# ============================================================

def cli_v4(question, model="gpt-4o-mini", system_prompt=None, 
           temperature=0.3, stream=True):
    """Full-featured CLI with all options."""
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": question})
    
    try:
        if stream:
            response_stream = client.chat.completions.create(
                model=model, messages=messages, 
                temperature=temperature, max_tokens=1000, stream=True
            )
            collected = []
            for chunk in response_stream:
                content = chunk.choices[0].delta.content
                if content:
                    print(content, end="", flush=True)
                    collected.append(content)
            print(f"\n\n{'─' * 50}")
            print(f"Model: {model} | temp: {temperature}")
            if system_prompt:
                print(f"System: {system_prompt[:60]}..." if len(system_prompt) > 60 else f"System: {system_prompt}")
        else:
            response = client.chat.completions.create(
                model=model, messages=messages,
                temperature=temperature, max_tokens=1000
            )
            print(response.choices[0].message.content)
            usage = response.usage
            print(f"\n{'─' * 50}")
            print(f"Model: {model} | Tokens: {usage.total_tokens} | temp: {temperature}")
    
    except Exception as e:
        print(f"\n❌ Error: {e}")


# Test with different configurations
print("═" * 60)
print("Default (no system prompt):")
print("═" * 60)
cli_v4("Review this code: def add(a,b): return a+b")

print(f"\n\n{'═' * 60}")
print("With security expert system prompt:")
print("═" * 60)
cli_v4(
    "Review this code: def add(a,b): return a+b",
    system_prompt="You are a senior security engineer. Focus on security vulnerabilities."
)

### The finished CLI tool

The final version is in `cli_tool/ask.py`. It adds:
- `argparse` for proper command-line arguments
- `--model`, `--system`, `--temperature`, `--stream`, `--stdin` flags
- Pipe support: `echo "some text" | python ask.py "Summarize this" --stdin`
- Full error handling with exponential backoff
- Token and cost tracking

Run it:
```bash
python cli_tool/ask.py "What is Hadoop?"
python cli_tool/ask.py "Review this code" --model gpt-4o --stream
python cli_tool/ask.py "Explain RAG" --system "You are a senior ML engineer"
cat some_file.txt | python cli_tool/ask.py "Summarize this" --stdin
```

---

---

# Chapter 6: Build — Streamlit App

---

*Course page: Chapter 6 — Build: Streamlit App*

Streamlit turns Python scripts into web apps. Let's build step by step.

**Note:** Streamlit apps can't run inside notebooks. We'll write the code here, but you run it from the terminal with `streamlit run app.py`.

### Version 1: Bare minimum chat app (15 lines)

In [ ]:
# ============================================================
# V1: Simplest Streamlit chat app
# ============================================================
# Save this as app_v1.py and run: streamlit run app_v1.py

v1_code = '''
import streamlit as st
from openai import OpenAI

client = OpenAI()
st.title("My GenAI App")

if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

if prompt := st.chat_input("Ask anything..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.write(prompt)
    
    with st.chat_message("assistant"):
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=st.session_state.messages
        )
        answer = response.choices[0].message.content
        st.write(answer)
    
    st.session_state.messages.append({"role": "assistant", "content": answer})
'''

print("V1: Simplest Streamlit chat app")
print("═" * 50)
print(v1_code)
print("\n💡 That's ~20 lines for a fully working chat app with conversation history.")
print("   Save as app_v1.py → run: streamlit run app_v1.py")

### Version 2: Add streaming

In [ ]:
# ============================================================
# V2: Add streaming — one line change
# ============================================================

v2_change = '''
# BEFORE (standard — user waits):
    with st.chat_message("assistant"):
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=st.session_state.messages
        )
        answer = response.choices[0].message.content
        st.write(answer)

# AFTER (streaming — tokens appear immediately):
    with st.chat_message("assistant"):
        stream = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=st.session_state.messages,
            stream=True                              # ← add this
        )
        answer = st.write_stream(                    # ← change st.write to st.write_stream
            chunk.choices[0].delta.content or ""
            for chunk in stream
            if chunk.choices[0].delta.content
        )
'''

print("V2: Adding streaming")
print("═" * 50)
print(v2_change)
print("\n💡 st.write_stream() handles everything — displays tokens as they arrive")
print("   AND returns the full text so you can save it to history.")

### Version 3: Add sidebar with settings

In [ ]:
# ============================================================
# V3: Sidebar with model selector + system prompt + temperature
# ============================================================

v3_sidebar = '''
with st.sidebar:
    st.markdown("### Settings")
    
    model = st.selectbox("Model", ["gpt-4o-mini", "gpt-4o"])
    
    system_prompt = st.text_area(
        "System Prompt",
        value="You are a helpful assistant.",
        height=100
    )
    
    temperature = st.slider(
        "Temperature", 0.0, 2.0, 0.7, 0.1
    )

# Then in your API call, use these variables:
api_messages = [{"role": "system", "content": system_prompt}]
api_messages.extend(st.session_state.messages)

stream = client.chat.completions.create(
    model=model,               # from sidebar
    messages=api_messages,      # includes system prompt
    temperature=temperature,    # from sidebar
    stream=True
)
'''

print("V3: Sidebar settings")
print("═" * 50)
print(v3_sidebar)

### Version 4: Add cost tracking

In [ ]:
# ============================================================
# V4: Cost tracking in session state
# ============================================================

v4_tracking = '''
# Initialize in session state:
if "total_tokens" not in st.session_state:
    st.session_state.total_tokens = 0
if "total_cost" not in st.session_state:
    st.session_state.total_cost = 0.0

# After each response, update:
est_tokens = (len(prompt) + len(response_text)) // 4
st.session_state.total_tokens += est_tokens
st.session_state.total_cost += est_tokens * 0.15e-6  # mini pricing

# Show in sidebar:
with st.sidebar:
    st.metric("Total tokens", f"{st.session_state.total_tokens:,}")
    st.metric("Estimated cost", f"${st.session_state.total_cost:.4f}")
'''

print("V4: Cost tracking")
print("═" * 50)
print(v4_tracking)
print("\n💡 st.metric() creates nice dashboard-style numbers in the sidebar.")

### The finished Streamlit app

The final version is in `streamlit_app/app.py`. It combines everything:
- Chat interface with conversation history
- Sidebar: model selector (GPT-4o-mini, GPT-4o, Llama via Groq), system prompt editor, temperature slider
- Streaming responses
- Token and cost tracking in sidebar
- Export conversation as JSON
- Clear chat button
- Error handling with retry on rate limits

Run it:
```bash
cd streamlit_app
pip install -r requirements.txt
streamlit run app.py
```

---

---

# Summary

---

| Chapter | What you learned | Key skill |
|---|---|---|
| **Error Handling** | APIs fail — handle it gracefully | Exponential backoff, finish_reason checks |
| **Streaming** | Show tokens as they generate | `stream=True` + iterate chunks |
| **Multi-Provider** | Same task, different providers | Provider-agnostic wrapper, fallback pattern |
| **App Architecture** | Notebooks ≠ applications | Layers: UI → Logic → LLM API → Data |
| **CLI Tool** | Build a terminal app step by step | argparse, retry, streaming, cost tracking |
| **Streamlit App** | Build a web app in ~50 lines | session_state, chat_input, write_stream |

### What's in the folder

```
3_notebooks_to_apps/
├── 1_Building_Apps.ipynb          ← this notebook (teaches the concepts)
├── cli_tool/
│   └── ask.py                     ← finished CLI tool
└── streamlit_app/
    ├── app.py                     ← finished Streamlit app
    └── requirements.txt
```

**Next → RAG (Retrieval Augmented Generation)** — connecting LLMs to your own data.